Woah sick

In [2]:
!pip install pyspark pyarrow

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import os
print(os.getcwd())

/expanse/lustre/projects/uci157/tragus


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DoubleType, FloatType
from pyspark.sql.functions import col, count, length,countDistinct, min as spark_min, max as spark_max,avg, stddev, approx_count_distinct

spark = (SparkSession.builder.appName("MusicBrainz").config("spark.driver.memory", "2g").config("spark.executor.memory", "18g").config('spark.executor.instances', 7).getOrCreate())

I saw on Piazza somebody was asking about the Spark Jobs UI, and apparently I was right that this is something distinct from the Expanse Jobs page. Apparently it's this-

Not sure how to get it to display the active number of tasks since it's not live

In [5]:
import requests
import pandas as pd

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
spark_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
spark_df['maxMemory_GB'] = (spark_df['maxMemory'] / (1024**3)).round(2)
spark_df

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,8,1099746508,0,True,1.02


In [6]:
MBDUMP = "musicbrainz_project/raw_data/mbdump"

In [7]:
def peek(df, n=20):
    return display(df.limit(n).toPandas())

# DEFINE SCHEMAS

In [8]:
artist_schema = StructType([
    StructField("id",               IntegerType(),   True),
    StructField("gid",              StringType(),    True),
    StructField("name",             StringType(),    True),
    StructField("sort_name",        StringType(),    True),
    StructField("begin_date_year",  IntegerType(),   True),
    StructField("begin_date_month", IntegerType(),   True),
    StructField("begin_date_day",   IntegerType(),   True),
    StructField("end_date_year",    IntegerType(),   True),
    StructField("end_date_month",   IntegerType(),   True),
    StructField("end_date_day",     IntegerType(),   True),
    StructField("type",             IntegerType(),   True),
    StructField("area",             IntegerType(),   True),
    StructField("gender",           IntegerType(),   True),
    StructField("comment",          StringType(),    True),
    StructField("edits_pending",    IntegerType(),   True),
    StructField("last_updated",     StringType(),    True),
    StructField("ended",            StringType(),    True),
    StructField("begin_area",       IntegerType(),   True),
    StructField("end_area",         IntegerType(),   True),
])
instrument_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("type",          IntegerType(), True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
    StructField("comment",       StringType(),  True),
    StructField("description",   StringType(),  True),
])
label_schema = StructType([
    StructField("id",                IntegerType(), True),
    StructField("gid",               StringType(),  True),
    StructField("name",              StringType(),  True),
    StructField("begin_date_year",   IntegerType(), True),
    StructField("begin_date_month",  IntegerType(), True),
    StructField("begin_date_day",    IntegerType(), True),
    StructField("end_date_year",     IntegerType(), True),
    StructField("end_date_month",    IntegerType(), True),
    StructField("end_date_day",      IntegerType(), True),
    StructField("label_code",        IntegerType(), True),
    StructField("type",              IntegerType(), True),
    StructField("area",              IntegerType(), True),
    StructField("comment",           StringType(),  True),
    StructField("edits_pending",     IntegerType(), True),
    StructField("last_updated",      StringType(),  True),
    StructField("ended",             StringType(),  True),
])
genre_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
area_schema = StructType([
    StructField("id",               IntegerType(), True),
    StructField("gid",              StringType(),  True),
    StructField("name",             StringType(),  True),
    StructField("type",             IntegerType(), True),
    StructField("edits_pending",    IntegerType(), True),
    StructField("last_updated",     StringType(),  True),
    StructField("begin_date_year",  IntegerType(), True),
    StructField("begin_date_month", IntegerType(), True),
    StructField("begin_date_day",   IntegerType(), True),
    StructField("end_date_year",    IntegerType(), True),
    StructField("end_date_month",   IntegerType(), True),
    StructField("end_date_day",     IntegerType(), True),
    StructField("ended",            StringType(),  True),
    StructField("comment",          StringType(),  True),
])
tag_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("ref_count",     IntegerType(), True),
])
gender_schema = StructType([
    StructField("id",   IntegerType(), True),
    StructField("name", StringType(),  True),
])
release_group_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("title",         StringType(),  True),
    StructField("artist_credit", IntegerType(), True),
    StructField("type",          IntegerType(), True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
l_artist_label_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # label
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
l_artist_release_group_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # release_group
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
label_tag_schema = StructType([
    StructField("label",        IntegerType(), True),
    StructField("tag",          IntegerType(), True),
    StructField("count",        IntegerType(), True),
    StructField("last_updated", StringType(),  True),
])
l_artist_genre_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # genre
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
l_artist_artist_schema = StructType([
    StructField("id",      IntegerType(), True),
    StructField("link",    IntegerType(), True),
    StructField("entity0", IntegerType(), True),
    StructField("entity1", IntegerType(), True),
])
release_group_tag_schema = StructType([
    StructField("release_group", IntegerType(), True),
    StructField("tag",           IntegerType(), True),
    StructField("count",         IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
artist_tag_schema = StructType([
    StructField("artist",       IntegerType(), True),
    StructField("tag",          IntegerType(), True),
    StructField("count",        IntegerType(), True),
    StructField("last_updated", StringType(),  True),
])
artist_credit_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("artist_count",  IntegerType(), True),
    StructField("ref_count",     IntegerType(), True),
    StructField("created",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("gid",           StringType(),  True),
])
artist_credit_name_schema = StructType([
    StructField("artist_credit", IntegerType(), True),
    StructField("position",      IntegerType(), True),
    StructField("artist",        IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("join_phrase",   StringType(),  True),
])
l_artist_instrument_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # instrument
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
link_schema = StructType([
    StructField("id",                IntegerType(), True),
    StructField("link_type",         IntegerType(), True),
    StructField("begin_date_year",   IntegerType(), True),
    StructField("begin_date_month",  IntegerType(), True),
    StructField("begin_date_day",    IntegerType(), True),
    StructField("end_date_year",     IntegerType(), True),
    StructField("end_date_month",    IntegerType(), True),
    StructField("end_date_day",      IntegerType(), True),
    StructField("attribute_count",   IntegerType(), True),
    StructField("created",           StringType(),  True),
    StructField("ended",             StringType(),  True),
])
link_type_schema = StructType([
    StructField("id",                  IntegerType(), True),
    StructField("parent",              IntegerType(), True),
    StructField("child_order",         IntegerType(), True),
    StructField("gid",                 StringType(),  True),
    StructField("entity_type0",        StringType(),  True),
    StructField("entity_type1",        StringType(),  True),
    StructField("name",                StringType(),  True),
    StructField("description",         StringType(),  True),
    StructField("link_phrase",         StringType(),  True),
    StructField("reverse_link_phrase", StringType(),  True),
    StructField("long_link_phrase",    StringType(),  True),
    StructField("last_updated",        StringType(),  True),
    StructField("is_deprecated",       StringType(),  True),
    StructField("has_dates",           StringType(),  True),
    StructField("attribute_count",     IntegerType(), True),
    StructField("priority",            IntegerType(), True),
])

So the schema definitely aren't perfect - after testing out some SQL queries it becomes clear some of variables aren't assigned to the right columns. I think it will take some trial and error to suss out which variables are assigned correctly. It's a really great start though!! I've adjusted a couple and I'll spend a little more time experimenting with SQL to improve it

Something to be aware of is that a lot of these schema don't have all of the columns present in the TSV. We can still create the dataframes bc if there are extra columns without assignment in the schema, Spark just ignores them. Usually the variables we care about appear in the first half of the dataframe so ... maybe not so important but just FYI

### update:
it turns out we were missing some important tables, link and link_type. All "l_" tables have a link column which contains a code that determines the type of relationship being described in that row, and the link table connects to "link_type" which has more in-depth metadata on the particular relationship. 

# BUILD TABLES

In [9]:
schemas = {
    "artist": artist_schema,
    "instrument": instrument_schema,
    "label": label_schema,
    "genre": genre_schema,
    "area": area_schema,
    "tag": tag_schema,
    "gender": gender_schema,
    "release_group": release_group_schema,
    "l_artist_label": l_artist_label_schema,
    "l_artist_release_group": l_artist_release_group_schema,
    "label_tag": label_tag_schema,
    "l_artist_genre": l_artist_genre_schema,
    "l_artist_artist": l_artist_artist_schema,
    "release_group_tag": release_group_tag_schema,
    "artist_tag": artist_tag_schema,
    "artist_credit": artist_credit_schema,
    "artist_credit_name": artist_credit_name_schema,
    "l_artist_instrument": l_artist_instrument_schema,
    "link": link_schema,
    "link_type": link_type_schema,
}

dfs = {}
profile_rows = []

for table_name, schema in schemas.items():
    df = (
        spark.read
        .option("sep", "\t")
        .option("nullValue", r"\N")
        .option("header", "false")
        .option("quote", "")
        .option("escape", "")
        .schema(schema)
        .csv(f"{MBDUMP}/{table_name}")
    )

    dfs[table_name] = df

    print(f"=== PEEK: {table_name} ===")
    peek(df)

=== PEEK: artist ===


,id,gid,name,sort_name,begin_date_year,begin_date_month,begin_date_day,end_date_year,end_date_month,end_date_day,type,area,gender,comment,edits_pending,last_updated,ended,begin_area,end_area
0,2252039,fadeb38c-833f-40bc-9d8c-a6383b38b1be,Доктор Сатана,Доктор Сатана,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2021-11-23 07:08:52.479537+00,f,NaN,NaN
1,371203,49add228-eac5-4de8-836c-d75cde7369c3,Pete Moutso,"Moutso, Pete",NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,None,0,None,f,NaN,NaN
2,3087346,dfdce491-133d-4e9f-9e48-795587e181b0,UNlT,UNlT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2025-09-14 23:53:28.004798+00,f,NaN,NaN
3,2851271,165a49a0-2b3b-4078-a3c1-905afdc07c0a,Babyglock,Babyglock,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2024-10-19 03:33:55.474151+00,f,NaN,NaN
4,145773,7b4a548e-a01a-49b7-82e7-b49efeb9732c,Aric Leavitt,"Leavitt, Aric",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN
5,1076328,60aca66f-e91a-4cb5-9308-b6e293cd833e,Fonograff,Fonograff,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2014-01-10 16:25:20.992213+00,f,NaN,NaN
6,1172876,3e1bd546-d2a7-49cb-b38d-d70904a1d719,Al Street,"Street, Al",NaN,NaN,NaN,NaN,NaN,NaN,1.0,222.0,1.0,None,0,2014-11-23 14:07:19.782509+00,f,NaN,NaN
7,220155,df120895-f6c6-4a66-b9cf-73350f0beb61,Love .45,Love .45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN
8,618464,c14f8d3f-ee81-416f-800f-8eff7e77a2e1,Sintellect,Sintellect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2009-05-23 09:41:53.269195+00,f,NaN,NaN
9,285714,b68a3969-319a-462f-942b-cd35581414fc,Evie Tamala,Evie Tamala,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN


=== PEEK: instrument ===


,id,gid,name,type,edits_pending,last_updated,comment,description
0,687,c1dbb66d-2356-417a-81ad-f688fee33257,guitarrón mexicano,2,0,2015-02-15 07:32:32.570132+00,None,The guitarrón mexicano is a very large and dee...
1,695,2474c241-d267-433a-a404-688b13c51d11,jouhikko,2,0,2015-02-25 19:47:04.610414+00,None,"The jouhikko is a traditional, 2 or 3 stringed..."
2,701,c0cc863c-ea65-4b8a-b365-28b81b72d846,friction idiophone,3,0,2015-02-26 10:28:49.766548+00,None,Friction idiophones are idiophones where the s...
3,706,5f9bb15a-738f-48bb-8676-55e62547726f,doshpuluur,2,0,2015-02-28 00:15:24.010482+00,None,The doshpuluur is a long-necked Tuvan lute.
4,707,9bae90ea-9729-4c30-bd3e-f8319cdf4051,igil,2,0,2015-02-28 23:39:39.01131+00,None,The igil is a Tuvan bowed string instrument wi...
5,712,c1d76a22-ab32-45af-be76-f3abd1eb2b2a,saw sam sai,2,0,2015-03-01 12:00:44.315438+00,None,The saw sam sai is a three-stringed bowed inst...
6,1003,50ae3eb1-0034-4a34-8367-aff6c30dee9a,gendèr panerus,3,0,2019-11-11 19:35:36.39114+00,Higest pitch metallophone used in Javanese gam...,"Used for simple ornamentation, it has thick sm..."
7,684,4e22ddb3-6908-4a5f-a9ae-b8a7440f6c7c,nabal,1,0,2026-02-19 21:37:08.325951+00,Korean straight brass horn,"Used in traditional music, it's long and strai..."
8,732,d3667988-a354-4fc4-9c1e-9baea9a43c06,bass oboe,1,0,2015-03-04 11:26:51.888966+00,None,The bass oboe is a double reed woodwind instru...
9,734,2ab4c035-374a-4188-bb7a-81c2558254f7,krar,2,0,2015-03-06 03:22:56.003792+00,None,The krar is a five or six-stringed bowl-shaped...


=== PEEK: label ===


...

,id,link,entity0,entity1,edits_pending,last_updated,link_order,entity0_credit,entity1_credit
0,1,311992,1218252,121,0,2016-08-01 11:22:19.207793+00,0,None,None
1,2,312066,1395197,19,0,2016-08-01 14:14:07.499778+00,0,None,None
2,24,393478,1261488,808,0,2017-05-31 08:31:04.355584+00,0,None,None
3,4,312066,1395235,118,0,2016-08-01 18:16:51.429318+00,0,None,None
4,5,312066,1395197,21,0,2016-08-01 18:18:38.812703+00,0,None,None
5,6,312066,173057,779,0,2016-08-01 18:21:47.746656+00,0,None,None
6,7,312066,1395236,737,0,2016-08-01 18:21:59.822953+00,0,None,None
7,8,312066,1395237,139,0,2016-08-01 18:24:33.983833+00,0,None,None
8,9,312096,1395244,778,0,2016-08-01 18:33:50.830055+00,0,None,None
9,62,451269,1595656,890,0,2017-12-12 21:15:53.856419+00,0,None,None


=== PEEK: link ===


,id,link_type,begin_date_year,begin_date_month,begin_date_day,end_date_year,end_date_month,end_date_day,attribute_count,created,ended
0,48067,148,NaN,NaN,NaN,NaN,NaN,NaN,2,2012-05-15 20:52:02.141676+00,f
1,2,6,NaN,NaN,NaN,NaN,NaN,NaN,0,2011-05-16 15:03:23.368437+00,f
2,3,2,NaN,NaN,NaN,NaN,NaN,NaN,0,2011-05-16 15:03:23.368437+00,f
3,4,2,NaN,NaN,NaN,NaN,NaN,NaN,1,2011-05-16 15:03:23.368437+00,f
4,5,1,NaN,NaN,NaN,NaN,NaN,NaN,0,2011-05-16 15:03:23.368437+00,f
5,6,1,NaN,NaN,NaN,NaN,NaN,NaN,1,2011-05-16 15:03:23.368437+00,f
6,30853,103,2011.0,10.0,NaN,NaN,NaN,NaN,0,2011-10-30 18:54:18.334591+00,f
7,8,9,NaN,NaN,NaN,NaN,NaN,NaN,0,2011-05-16 15:03:23.368437+00,f
8,9,15,NaN,NaN,NaN,NaN,NaN,NaN,0,2011-05-16 15:03:23.368437+00,f
9,10,17,NaN,NaN,NaN,NaN,NaN,NaN,0,2011-05-16 15:03:23.368437+00,f


=== PEEK: link_type ===


,id,parent,child_order,gid,entity_type0,entity_type1,name,description,link_phrase,reverse_link_phrase,long_link_phrase,last_updated,is_deprecated,has_dates,attribute_count,priority
0,12,NaN,0,38278b3b-30e6-304c-b0db-5ba701eb0268,release_group,release_group,covers and versions,None,covers or other versions,covers or other versions,covers and versions,2014-05-18 09:46:23.72719+00,f,f,0,0
1,735,NaN,0,12678b88-1adb-3536-890e-9b39b9a14b2d,instrument,instrument,children,None,children,child of,has child,2014-05-18 10:41:05.403719+00,f,f,0,0
2,870,784.0,0,4789521b-57b9-4689-9644-46de63190f66,series,url,soundcloud,"This links a series (most commonly, but not ne...",SoundCloud,SoundCloud page for,has a SoundCloud page at,2015-12-16 11:53:10.160133+00,f,t,0,0
3,957,NaN,2,a6874915-b6a3-42ef-82af-6ba800d1e940,label,url,get the music,None,get the music,get the music,get the music,2018-05-18 10:37:46.114037+00,f,f,0,0
4,227,234.0,0,451076df-61cf-46ab-9921-555cab2f050d,recording,recording,DJ-mix,"This is used to link a <a href=""/doc/Mix_Termi...",DJ-mix of,DJ-mixes,is a DJ-mix of,2021-02-02 10:14:29.683106+00,f,t,0,0
5,754,188.0,0,8147b6a2-ad14-4ce7-8f0a-697f9a31f68f,artist,url,IMSLP,"This links an artist to its page in <a href=""h...",IMSLP,IMSLP page for,has an IMSLP page at,2020-02-17 13:17:20.780295+00,f,t,0,0
6,921,NaN,1,9900c8c5-9844-4f10-8403-23aabafd913c,url,work,work list entry,This link points to a page for a particular wo...,work list entry for,work list entry,{entity1} has a work list entry at {entity0},2019-08-14 15:57:32.555087+00,f,t,0,0
7,2,4.0,0,fc399d47-23a7-4c28-bfcf-0607a562b644,release,release,transl-tracklisting,This indicates that one release is identical t...,transliterated/translated track listings,transliterated/translated track listing of,is the original for the transliterated/transla...,2015-07-17 08:53:57.281375+00,f,f,0,0
8,154,157.0,1,83f72956-2007-4bca-8a97-0ae539cca99d,artist,recording,samples from artist,Indicates that the recording contains samples ...,produced material that was {additional:additio...,contains {additional} samples by,{entity1} contains {additional} samples by {en...,2015-11-05 16:48:09.775478+00,f,t,1,0
9,871,54.0,3,4db37fec-eb67-45d3-b4fa-148a68135fbb,artist,release,translator,Indicates the person who translated the lyrics...,translated,{additional} translator,{additional:additionally} translated,2015-08-18 22:41:18.220935+00,f,t,1,0


# FOREIGN KEY MAP (for navigating the database)

In [10]:
foreign_key_map = {
    "artist": {
        "type":       ("artist_type", "id"),
        "area":       ("area", "id"),
        "gender":     ("gender", "id"),
        "begin_area": ("area", "id"),
        "end_area":   ("area", "id"),
    },
    "instrument": {
        "type": ("instrument_type", "id"),
    },
    "label": {
        "type": ("label_type", "id"),
        "area": ("area", "id"),
    },
    "release_group": {
        "type":          ("release_group_primary_type", "id"),
        "artist_credit": ("artist_credit", "id"),
    },
    "artist_credit_name": {
        "artist_credit": ("artist_credit", "id"),
        "artist":        ("artist", "id"),
    },
    "label_tag": {
        "label": ("label", "id"),
        "tag":   ("tag", "id"),
    },
    "artist_tag": {
        "artist": ("artist", "id"),
        "tag":    ("tag", "id"),
    },
    "release_group_tag": {
        "release_group": ("release_group", "id"),
        "tag":           ("tag", "id"),
    },
    "l_artist_genre": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("genre", "id"),
    },
    "l_release_group_genre": { 
        "link":    ("link", "id"),
        "entity0": ("release_group", "id"),
        "entity1": ("genre", "id"),
    },
    "l_artist_label": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("label", "id"),
    },
    "l_artist_release_group": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("release_group", "id"),
    },
    "l_artist_artist": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("artist", "id"),
    },
    "l_artist_instrument": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("instrument", "id"),
    },
    "link": {
        "link_type": ("link_type", "id"),
    },
    "link_type": {
        "parent": ("link_type", "id"),
    },
}

Great work on the map! I just removed "artist_genre" and "release_group_genre" because I couldn't find those tables in the database. But I'm pretty sure it's correct

# TESTING SOME BASIC SQL QUERIES

Here are some cool queries

In [11]:
for table_name, df in dfs.items():
    df.createOrReplaceTempView(table_name)

In [12]:
def peek(df, n=20):
    return display(df.limit(n).toPandas())

***

1) Select all from artist

In [13]:
peek(spark.sql("""
    SELECT *
    FROM artist
"""))

,id,gid,name,sort_name,begin_date_year,begin_date_month,begin_date_day,end_date_year,end_date_month,end_date_day,type,area,gender,comment,edits_pending,last_updated,ended,begin_area,end_area
0,2252039,fadeb38c-833f-40bc-9d8c-a6383b38b1be,Доктор Сатана,Доктор Сатана,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2021-11-23 07:08:52.479537+00,f,NaN,NaN
1,371203,49add228-eac5-4de8-836c-d75cde7369c3,Pete Moutso,"Moutso, Pete",NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,None,0,None,f,NaN,NaN
2,3087346,dfdce491-133d-4e9f-9e48-795587e181b0,UNlT,UNlT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2025-09-14 23:53:28.004798+00,f,NaN,NaN
3,2851271,165a49a0-2b3b-4078-a3c1-905afdc07c0a,Babyglock,Babyglock,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2024-10-19 03:33:55.474151+00,f,NaN,NaN
4,145773,7b4a548e-a01a-49b7-82e7-b49efeb9732c,Aric Leavitt,"Leavitt, Aric",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN
5,1076328,60aca66f-e91a-4cb5-9308-b6e293cd833e,Fonograff,Fonograff,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2014-01-10 16:25:20.992213+00,f,NaN,NaN
6,1172876,3e1bd546-d2a7-49cb-b38d-d70904a1d719,Al Street,"Street, Al",NaN,NaN,NaN,NaN,NaN,NaN,1.0,222.0,1.0,None,0,2014-11-23 14:07:19.782509+00,f,NaN,NaN
7,220155,df120895-f6c6-4a66-b9cf-73350f0beb61,Love .45,Love .45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN
8,618464,c14f8d3f-ee81-416f-800f-8eff7e77a2e1,Sintellect,Sintellect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2009-05-23 09:41:53.269195+00,f,NaN,NaN
9,285714,b68a3969-319a-462f-942b-cd35581414fc,Evie Tamala,Evie Tamala,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN


***

2. Select all artists and their labels

In [14]:
peek(spark.sql("""
    SELECT a.name AS artist_name, l.name AS label
    FROM l_artist_label lal
    JOIN artist a ON lal.entity0 = a.id
    JOIN label l ON lal.entity1 = l.id
"""))

,artist_name,label
0,s'Poom,Legendaarne Records
1,Guido Elmi,Nopop
2,Marco Resmann,Upon.You
3,Karl-Jonas Winqvist,Sing a Song Fighter
4,Stuart Brown,British Film Institute
5,Earl Young,"Baker, Harris & Young Productions"
6,Alexia Coley,Jalapeno Records
7,Darren Hickey,Xpressive
8,Darren Hickey,Juice Records
9,Ras Sheehama,African Cream Music


***

3. Number of release groups per artist sorted by most prolific

In [15]:
peek(spark.sql("""
    SELECT a.name AS artist_name, COUNT(rg.id) AS release_group_count
    FROM release_group rg
    JOIN artist_credit_name acn ON rg.artist_credit = acn.artist_credit
    JOIN artist a ON acn.artist = a.id
    GROUP BY a.name
    ORDER BY release_group_count DESC
"""))

,artist_name,release_group_count
0,Various Artists,281842
1,Ludwig van Beethoven,6088
2,Johann Sebastian Bach,6011
3,Wolfgang Amadeus Mozart,5922
4,[unknown],3432
5,Johannes Brahms,2946
6,Franz Schubert,2852
7,Пётр Ильич Чайковский,2684
8,Antonio Vivaldi,2143
9,Bruce Springsteen,2127


***

4. Number of labels per country

In [16]:
peek(spark.sql("""
    SELECT a.name as area, COUNT(l.id) as label_count
    FROM label l
    JOIN area a ON l.area = a.id
    GROUP BY a.name
    ORDER BY label_count DESC
    LIMIT 20
"""))

,area,label_count
0,United States,30680
1,United Kingdom,14758
2,Japan,12059
3,Germany,9388
4,France,5974
5,Italy,4110
6,Canada,3227
7,Sweden,2933
8,Netherlands,2893
9,Spain,2345


***

5. Number of artists by gender

In [17]:
peek(spark.sql("""
    SELECT g.name AS gender, COUNT(*) AS count
    FROM artist a
    JOIN gender g ON a.gender = g.id
    GROUP BY g.name
    ORDER BY count DESC
"""))

,gender,count
0,Male,932192
1,Female,280203
2,Not applicable,2443
3,Non-binary,2237
4,Other,1787


***

6. Relationships between artists and their labels

In [18]:
peek(spark.sql("""
    SELECT 
    a.name as artist,
    l.name as label,
    lt.name as relationship_type,
    lnk.begin_date_year,
    lnk.end_date_year
    FROM l_artist_label lal
    JOIN artist a ON lal.entity0 = a.id
    JOIN label l ON lal.entity1 = l.id
    JOIN link lnk ON lal.link = lnk.id
    JOIN link_type lt ON lnk.link_type = lt.id
"""))

,artist,label,relationship_type,begin_date_year,end_date_year
0,Murdock,Radar Records,label founder,2010,NaN
1,Empty Pools,Battle Worldwide Recordings,recording contract,2011,NaN
2,Dr. Jekyll,Hirntot Records,recording contract,2006,2011.0
3,eXiled,Foundry Music,recording contract,2013,NaN
4,Ben Kass,Underground Music,creative position at,2009,NaN
5,Iron Mike Norton,GFO Records,producer position at,2007,NaN
6,Social Distortion,Restless Records,recording contract,1988,1989.0
7,Excel,Caroline Records,recording contract,1988,1989.0
8,Henry Burr,Henry Burr Music Corp.,label founder,1919,1919.0
9,Dan Miracle,Reinforced Records,producer position at,2003,2004.0


***

7. Possible types of relationships that can exist between an artist and a label

In [19]:
peek(spark.sql("""
    SELECT id, name, link_phrase
    FROM link_type
    WHERE entity_type0 = 'artist' AND entity_type1 = 'label'
"""))

,id,name,link_phrase
0,1258,named after artist,inspired the name of label
1,119,position at,employed by
2,1259,named after label,named after label
3,724,personal publisher,has personal publisher
4,115,creative position at,creative position
5,990,ownership,ownership
6,116,label founder,founded
7,120,engineer position at,engineer position
8,1081,artists and repertoire position at,artists and repertoire position
9,723,personal label,has personal label


***

Some sanity checking procedures...

In [22]:
peek(spark.sql("""
    SELECT COUNT(*) as matched
    FROM l_artist_label lal
    JOIN artist a ON lal.entity0 = a.id
    JOIN label l ON lal.entity1 = l.id
"""))

peek(spark.sql("""
    SELECT COUNT(*) FROM l_artist_label
"""))

,matched
0,64053


,count(1)
0,64053


In [26]:
peek(spark.sql("""
    SELECT COUNT(*) as matched
    FROM artist_tag at
    JOIN tag t ON at.tag = t.id
"""))

peek(spark.sql("""
    SELECT COUNT(*) as total FROM artist_tag
"""))

,matched
0,724177


,total
0,724177


Nulls analysis

In [24]:
peek(spark.sql("""
    SELECT
        SUM(CASE WHEN type IS NULL THEN 1 ELSE 0 END) as null_type,
        SUM(CASE WHEN area IS NULL THEN 1 ELSE 0 END) as null_area,
        SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END) as null_gender,
        COUNT(*) as total
    FROM artist
"""))

,null_type,null_area,null_gender,total
0,638626,1438400,1634292,2853154


***

# Part 2, #5: Preprocessing Plan

Our first milestone will be to build a K-Nearest Neighbors which groups music based on its "DNA" (instruments, genres, labels, etc), so the first step will be to determine precisely which variables constitute part of music's DNA, and determining their relative importance. The next step will be to strip away unnecessary information, such as columns and tables we won't need (for example: "description" in the instruments table, since it contains natural language, will not be helpful for this problem). 

The data has a high volume of nulls. Depending on the data type these will need to be dealt with differently - for example, date columns might be left as nulls because imputing an "average" date probably doesn't make any sense. "Type" columns (which indicate something else for each table, but can refer to things like release type: original, bootleg, reissue, etc) have their most common entry set to 1, so nulls in that variable will likely be set to 1. There are also lots of entries in which most data is null - since the dataset is so large, some sets of rows might be filtered out completely.

We will primarily use SQL for preparing the data that we will operate on, then switch over to Python for filling nulls and cleaning. Some Spark transformations we may use include join, groupBy, collect_list, StringIndexer, OneHotEncoder, CountVectorizer, VectorAssembler, Normalizer, and filter.